In [1]:

# ============================================================
# Shared setup (anonymized; paths are never printed)
# ============================================================
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from scipy.stats import spearmanr

# Grayscale, publication style; no colour, no titles/captions in saved images.
sns.set_theme(style="whitegrid", context="paper")
plt.rcParams.update({"image.cmap": "gray", "axes.grid": True, "grid.alpha": 0.25,
                     "font.size": 11, "savefig.dpi": 600})

DATA = Path("../data"); FIG = Path("../results/figures"); TAB = Path("../results/tables")
RNG = np.random.default_rng(42)

# Dataset registry (public sources) and decision domain.
DATASETS = {"telco": "telco_churn.csv", "adult": "adult.parquet",
            "credit_risk": "credit_risk.csv", "gmsc": "gmsc.parquet",
            "taiwan": "taiwan.parquet"}
DOMAIN = {"telco": "non-credit", "adult": "non-credit",
          "credit_risk": "consumer credit", "gmsc": "consumer credit",
          "taiwan": "consumer credit"}
# Grayscale shade + marker per domain (colour-free, print-safe).
STYLE = {"consumer credit": ("0.15", "o"), "corporate credit": ("0.0", "*"),
         "non-credit": ("0.45", "s")}
# Paper-1 anchor (Polish corporate bankruptcy; values reported in the first paper).
POLISH = {"gini": 2*0.778-1, "pbar": 0.0482, "naive_gap": 2.72, "indep_gap": 0.33}

def load(name):
    f = DATA / DATASETS[name]
    return pd.read_parquet(f) if str(f).endswith(".parquet") else pd.read_csv(f)

def prep(name, df):
    # Return numeric design matrix, binary outcome, and two pd-independent drivers (b1,b2).
    df = df.copy()
    if name == "telco":
        y = (df["Churn"].astype(str).str.strip() == "Yes").astype(int).values
        df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
        X = df.drop(columns=["customerID", "Churn"]); b2col = "TotalCharges"
    elif name == "adult":
        y = df["HIGH_INCOME"].astype(int).values
        drop = ["income", "HIGH_INCOME", "AGE_YEARS", "SEX_STD", "EDU_STD", "VIP", "fnlwgt"]
        X = df.drop(columns=[c for c in drop if c in df]); b2col = "hours-per-week"
    elif name == "credit_risk":
        y = df["loan_status"].astype(int).values
        X = df.drop(columns=["loan_status"]); b2col = "person_income"
    elif name == "gmsc":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP"] if c in df]); b2col = "INCOME"
    elif name == "taiwan":
        y = df["DEFAULT"].astype(int).values
        X = df.drop(columns=[c for c in ["DEFAULT", "VIP", "AGE_YEARS", "SEX_STD"] if c in df]); b2col = "LIMIT_BAL"
    b2 = pd.to_numeric(df[b2col], errors="coerce").values if b2col in df else None
    Xn = pd.get_dummies(X, dummy_na=False).apply(pd.to_numeric, errors="coerce")
    return Xn, y, b2

def system_pd(X, y):
    # L2 logistic system model; out-of-fold PD via 5-fold stratified CV; in-fold median imputation.
    from sklearn.linear_model import LogisticRegression
    from sklearn.preprocessing import StandardScaler
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score
    pdh = np.zeros(len(y)); skf = StratifiedKFold(5, shuffle=True, random_state=42)
    for tr, te in skf.split(X, y):
        Xtr, Xte = X.iloc[tr].copy(), X.iloc[te].copy()
        med = Xtr.median(numeric_only=True); Xtr = Xtr.fillna(med); Xte = Xte.fillna(med)
        sc = StandardScaler().fit(Xtr)
        clf = LogisticRegression(C=0.1, class_weight="balanced", max_iter=2000)
        clf.fit(sc.transform(Xtr), y[tr]); pdh[te] = clf.predict_proba(sc.transform(Xte))[:, 1]
    return pdh, roc_auc_score(y, pdh)

def pick_b1(X, pdh):
    # Near-orthogonal driver: numeric feature with the lowest |Spearman| vs PD (Attr43 analogue).
    best, br = None, 2.0
    for c in X.columns:
        v = X[c].values.astype(float); m = ~np.isnan(v)
        if m.sum() < 100 or np.nanstd(v) == 0: continue
        r = abs(spearmanr(v[m], pdh[m]).statistic)
        if not np.isnan(r) and r < br: br, best = r, v
    return best

QGRID = [0, 0.10, 0.20, 0.35, 0.55, 0.75, 0.90, 1.0]
def assign_grades(pdh):
    qs = np.quantile(pdh, QGRID); qs[0] -= 1e-9; qs[-1] += 1e-9
    return np.digitize(pdh, qs[1:-1])
SCHED = np.array([0.10, 0.15, 0.20, 0.35, 0.50, 0.70, 0.80])  # ord 0 best .. ord 6 worst
OVR = 0.35

def rank01(v):
    return pd.Series(v).rank(pct=True).fillna(0.5).values

def expected_gap(pi, y):
    # Exact expected upgrade-minus-none default-rate gap (percentage points).
    return (np.sum(pi * y) / np.sum(pi) - y.mean()) * 100.0

def sim_once(pi_firm, grade, G, rng):
    # One generative override draw; returns signed grade_diff (negative = upgrade) and masks.
    N = len(grade)
    overridden = rng.random(N) < OVR
    up = overridden & (rng.random(N) < pi_firm)
    down = overridden & ~up
    notch = np.where(rng.random(N) < 0.70, 1, 2)
    gd = np.zeros(N, int); gd[up] = -notch[up]; gd[down] = notch[down]
    final = np.clip(grade + gd, 0, G - 1)
    return final - grade, overridden, up

def save_fig(fig, name):
    # Save grayscale figure as PNG and PDF at 600 dpi; no caption; no path printed.
    fig.savefig(FIG / (name + ".png"), dpi=600, bbox_inches="tight")
    fig.savefig(FIG / (name + ".pdf"), bbox_inches="tight")
    plt.close(fig)


# 02_dgm_replication.ipynb -- three DGMs (naive/b1/b2), single-draw gaps, 500-rep gap + OR
from sklearn.linear_model import LogisticRegression
single, rep, orr = [], [], []
for name in DATASETS:
    sc = pd.read_parquet(TAB / ("scores_" + name + ".parquet"))
    y = sc["y"].values; pdh = sc["pd"].values; grade = sc["grade"].values.astype(int)
    G = 7; pbar = y.mean()
    pis = {"naive": SCHED[grade],
           "b1": 0.10 + 0.70*rank01(sc["b1"].values),
           "b2": 0.10 + 0.70*rank01(sc["b2"].values)}
    for dgm, pif in pis.items():
        # single draw
        rng = np.random.default_rng(7)
        ov = rng.random(len(y)) < OVR; up = ov & (rng.random(len(y)) < pif); none = ~ov
        single.append({"dataset": name, "dgm": dgm, "n_up": int(up.sum()),
                       "DR_none_pct": round(100*y[none].mean(), 2),
                       "DR_up_pct": round(100*y[up].mean(), 2),
                       "gap_pp": round(100*(y[up].mean()-y[none].mean()), 3)})
        # 500 replications of the gap
        gaps = np.empty(500)
        for i in range(500):
            ov = RNG.random(len(y)) < OVR; up = ov & (RNG.random(len(y)) < pif); none = ~ov
            gaps[i] = 100*(y[up].mean() - y[none].mean())
        rep.append({"dataset": name, "dgm": dgm, "gap_mean": round(gaps.mean(), 3),
                    "gap_lo": round(np.percentile(gaps, 2.5), 3),
                    "gap_hi": round(np.percentile(gaps, 97.5), 3),
                    "excludes_0": not (np.percentile(gaps,2.5) <= 0 <= np.percentile(gaps,97.5))})
        # 200 replications of grade_diff OR controlling for PD
        ors = np.empty(500)
        for i in range(500):
            gd, _, _ = sim_once(pif, grade, G, RNG)
            clf = LogisticRegression(solver="liblinear", max_iter=200).fit(
                np.column_stack([gd, pdh]), y)
            ors[i] = np.exp(clf.coef_[0, 0])
        orr.append({"dataset": name, "dgm": dgm, "OR_mean": round(ors.mean(), 3),
                    "OR_lo": round(np.percentile(ors, 2.5), 3),
                    "OR_hi": round(np.percentile(ors, 97.5), 3),
                    "excludes_1": not (np.percentile(ors,2.5) <= 1 <= np.percentile(ors,97.5))})
single = pd.DataFrame(single); rep = pd.DataFrame(rep); orr = pd.DataFrame(orr)
single.to_csv(TAB / "dgm_single_draw.csv", index=False)
rep.to_csv(TAB / "replication_gap.csv", index=False)
orr.to_csv(TAB / "replication_or.csv", index=False)
# grayscale: 500-rep gap by dataset x dgm (left) and OR by dgm (right)
greys = {"naive": "0.2", "b1": "0.55", "b2": "0.8"}
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
ax = axes[0]
x = np.arange(len(DATASETS)); w = 0.26
for k, dgm in enumerate(["naive", "b1", "b2"]):
    sub = rep[rep.dgm == dgm].set_index("dataset").loc[list(DATASETS)]
    ax.bar(x + (k-1)*w, sub["gap_mean"], w, color=greys[dgm], edgecolor="k", linewidth=0.5, label=dgm)
    ax.errorbar(x + (k-1)*w, sub["gap_mean"],
                yerr=[sub["gap_mean"]-sub["gap_lo"], sub["gap_hi"]-sub["gap_mean"]],
                fmt="none", ecolor="k", capsize=2, linewidth=0.7)
ax.axhline(0, color="k", lw=0.8); ax.set_xticks(x); ax.set_xticklabels(list(DATASETS), rotation=20)
ax.set_ylabel("Upgrade - none gap (pp)"); ax.legend(title="DGM", fontsize=8)
ax = axes[1]
for k, dgm in enumerate(["naive", "b1", "b2"]):
    sub = orr[orr.dgm == dgm].set_index("dataset").loc[list(DATASETS)]
    ax.errorbar(sub["OR_mean"], x + (k-1)*w,
                xerr=[sub["OR_mean"]-sub["OR_lo"], sub["OR_hi"]-sub["OR_mean"]],
                fmt="o", color=greys[dgm], ecolor="0.6", capsize=2, ms=5, label=dgm)
ax.axvline(1.0, color="k", lw=0.8, ls="--"); ax.set_yticks(x); ax.set_yticklabels(list(DATASETS))
ax.set_xlabel("grade_diff odds ratio (control PD)"); ax.legend(title="DGM", fontsize=8)
save_fig(fig, "fig02_dgm_effects")
rep


,dataset,dgm,gap_mean,gap_lo,gap_hi,excludes_0
0,telco,naive,13.271,9.588,16.357,True
1,telco,b1,-0.139,-2.938,2.685,False
2,telco,b2,-4.612,-7.482,-2.031,True
3,adult,naive,14.733,13.494,15.935,True
4,adult,b1,0.003,-0.998,0.941,False
5,adult,b2,4.862,3.839,6.030,True
6,credit_risk,naive,12.748,11.243,14.230,True
7,credit_risk,b1,0.037,-1.209,1.295,False
8,credit_risk,b2,-5.037,-6.074,-3.930,True
9,gmsc,naive,3.590,3.121,3.965,True
